In [72]:
import pandas as pd
import sys
sys.path.insert(1, '../../scripts')
from utils import parameters as params

In [3]:
psim_me = pd.read_csv('/data2/hratch/human_me/processed/' + 'corrected_psim_me.csv', index_col = 0) 


In [34]:
res = pd.DataFrame()
res['premrna_counts'] = psim_me.PREMRNA_SEQ.dropna().apply(lambda x: {ntp: x.count(ntp)for ntp in set(x)})
res['premrna_length'] = psim_me.PREMRNA_SEQ.dropna().apply(lambda x: len(x))
res['premrna_prop'] = res.apply(lambda x: {k: v/x.premrna_length for k,v in x.premrna_counts.items()}, axis = 1)

premrna_L = res['premrna_length'].median()
premrna_avg_prop = {ntp: res['premrna_prop'].apply(lambda x: x[ntp]).median() for ntp in ['A', 'U', 'C', 'G']}

premrna_seq = ''
for ntp in ['A', 'U', 'C', 'G']:
    premrna_seq += ntp*round(premrna_avg_prop[ntp]*premrna_L)
    

res = pd.DataFrame()
res['mrna_counts'] = psim_me.MRNA_SEQ.dropna().apply(lambda x: {ntp: x.count(ntp)for ntp in set(x)})
res['mrna_length'] = psim_me.MRNA_SEQ.dropna().apply(lambda x: len(x))
res['mrna_prop'] = res.apply(lambda x: {k: v/x.mrna_length for k,v in x.mrna_counts.items()}, axis = 1)

mrna_L = res['mrna_length'].median()
mrna_avg_prop = {ntp: res['mrna_prop'].apply(lambda x: x[ntp]).median() for ntp in ['A', 'U', 'C', 'G']}

mrna_seq = ''
for ntp in ['A', 'U', 'C', 'G']:
    mrna_seq += ntp*round(mrna_avg_prop[ntp]*mrna_L)

res = pd.DataFrame()
res['protein_counts'] = psim_me.PROTEIN_SEQ.dropna().apply(lambda x: {ntp: x.count(ntp)for ntp in set(x)})
res['protein_length'] = psim_me.PROTEIN_SEQ.dropna().apply(lambda x: len(x))
res['protein_prop'] = res.apply(lambda x: {k: v/x.protein_length for k,v in x.protein_counts.items()}, axis = 1)

protein_L = round(mrna_L)/3

def get_prop(x, aa):
    if aa in x.keys():
        return x[aa]
    else:
        return 0
        
protein_avg_prop = {aa: res['protein_prop'].apply(lambda x: get_prop(x, aa)).median() for aa in params.amino_acids}

protein_seq = ''
for aa in params.amino_acids:
    protein_seq += aa*round(protein_avg_prop[aa]*protein_L)

build_files_path = '/data2/hratch/human_me/build_files/'
with open(build_files_path + 'dummy_protein_features.tab', 'w') as f:
    for seq_type, seq in {'premrna_seq': premrna_seq, 'mrna_seq': mrna_seq, 'protein_seq': protein_seq}.items():
        f.write(seq_type + '\t' + seq + '\n')